In [107]:
import os
import re
import json
import random
import pickle
import numpy as np
import pandas as pd
from rich.pretty import pprint
import matplotlib.pyplot as plt
from collections import defaultdict
from more_itertools import unique_justseen
from aymurai.database.utils import text_to_uuid
from aymurai.api.endpoints.routers.misc.document_extract import extraction

docs2analize = ['2','3','4','6','7','8']
IS_LLM_FILE = True # or False
LLM_PREDICTION_FILE = "predictions_openai-alldocsv1.pkl"
AYMURAI_JSONS_PATH = '/Users/sofi/Desktop/collectiveai/projects/AymurAI/' #'/Users/sofi/Desktop/collectiveAI/projects/AymurAI' # '/Users/sofi/Desktop/collectiveai/projects/AymurAI/'
DOCS_PATH = '/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/resources/data/sample/' #'/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/' 

In [105]:
def get_paragraph_predictions(row,predictions):
    preds = predictions.get(row['name'], [])
    return [
        p for p in preds
        if p['start_char'] >= (row['start_char']) and (p['end_char'] <= row['end_char'])
    ]

def take_start_end_paragraphs(paragraphs):
    # Create start and end character positions
    start_end_chars = []
    current_pos = 0

    for i, paragraph in enumerate(paragraphs):
        start_char = current_pos
        end_char = start_char + len(paragraph)
        start_end_chars.append(
            {
                "paragraph_position": i,
                "text": paragraph,
                "paragraph_id": str(text_to_uuid(paragraph)).replace("-", ""),
                "start_char": start_char,
                "end_char": end_char,
            }
        )
        # +1 for the newline character between paragraphs (except after the last one)
        current_pos = end_char + 1

    return start_end_chars

def constract_paragraph(document):
    paragraphs = [line.strip() for line in document.split("\n") if line.strip()]
    paragraphs = [re.sub(r"\s{2,}", " ", line) for line in paragraphs]
    paragraphs = list(unique_justseen(paragraphs))
    return paragraphs


def get_paragraph_predictions(row, predictions):
    preds = predictions.get(row['name'], [])
    return [
        p for p in preds
        if p['start_char'] >= row['start_char'] and p['end_char'] <= row['end_char']
    ]

def are_all_par(pars,df):
    return len(set(pars) - set(df.text_y.iloc[:])) == 0 

def sample_cases(df, label, model="prediction", n=3):
    # nombre dinámico para el campo de predicción principal
    pred_key = "openai" if model == "prediction" else ("ner" if model == "NER_prediction" else "predmodel")

    def _norm_label_text(item):
        if not isinstance(item, dict):
            return None, ""
        attrs = item.get("attrs", {}) or {}
        lab   = attrs.get("aymurai_label") or item.get("label")
        txt   = attrs.get("aymurai_alt_text") or item.get("extraction_text") or item.get("text", "")
        return lab, txt

    cases = {"TP": [], "FP": [], "FN": []}

    for _, row in df.iterrows():
        # validation
        val_raw = eval(row["validation"]) if row["validation"] else []
        val_spans = [_norm_label_text(v) for v in val_raw if isinstance(v, dict)]
        val_spans = [(lab, txt) for lab, txt in val_spans if lab is not None]

        # pred principal (según 'model')
        if model == "NER_prediction":
            pred_raw = eval(row[model]) if row.get(model) else []
        else:
            pred_raw = row.get(model) if row.get(model) else []
        pred_spans = [_norm_label_text(p) for p in pred_raw if isinstance(p, dict)]
        pred_spans = [(lab, txt) for lab, txt in pred_spans if lab is not None]

        # pred de ambos modelos (si existen las columnas)
        openai_raw = row.get("prediction")
        ner_raw    = row.get("NER_prediction")
        openai_list = openai_raw if (openai_raw and not isinstance(openai_raw, str)) else (eval(openai_raw) if openai_raw else [])
        ner_list    = ner_raw if (ner_raw and not isinstance(ner_raw, str)) else (eval(ner_raw) if ner_raw else [])

        openai_spans = [_norm_label_text(p) for p in (openai_list or []) if isinstance(p, dict)]
        openai_spans = [(lab, txt) for lab, txt in openai_spans if lab is not None]
        ner_spans    = [_norm_label_text(p) for p in (ner_list or []) if isinstance(p, dict)]
        ner_spans    = [(lab, txt) for lab, txt in ner_spans if lab is not None]

        # listas planas por label
        val_labels  = [lab for lab, _ in val_spans]
        pred_labels = [lab for lab, _ in pred_spans]

        # snippet e info extra
        text     = (row.get("text_x") or "")#[:300]
        para_id  = row.get("paragraph_id", None)
        doc_name = row.get("name", None)

        # armar registro con val + ambas preds + clave dinámica
        base_rec = {
            "paragraph_id": para_id,
            "document": doc_name,
            "parrafo": text,
            "val":   [t for lab, t in val_spans if lab == label],
            "openai": [t for lab, t in openai_spans if lab == label],
            "ner":    [t for lab, t in ner_spans    if lab == label],
        }
        # setear pred principal bajo su clave dinámica
        base_rec[pred_key] = [t for lab, t in pred_spans if lab == label]

        # clasificar TP/FP/FN
        if (label in val_labels) and (label in pred_labels):
            cases["TP"].append(base_rec)
        elif (label in val_labels) and (label not in pred_labels):
            cases["FN"].append(base_rec)
        elif (label not in val_labels) and (label in pred_labels):
            cases["FP"].append(base_rec)

    # muestra aleatoria top-n por tipo
    return {k: random.sample(v, min(len(v), n)) for k, v in cases.items() if v}



# Prepare dataframe

## Join databased .json (NER)

In [ ]:
json_files = [f for f in os.listdir(AYMURAI_JSONS_PATH) if f.endswith('.json')]
para_files = [f for f in json_files if f.startswith('anonymization_paragraph')]
para_df = pd.concat([pd.read_json(AYMURAI_JSONS_PATH + fname) for fname in para_files], ignore_index=True)

doc_files = [f for f in json_files if f.startswith('anonymization_document') and not 'paragraph' in f]
doc_df = pd.concat([pd.read_json(AYMURAI_JSONS_PATH + fname) for fname in doc_files], ignore_index=True)

doc_para_files = [f for f in json_files if f.startswith('anonymization_document_paragraph')]
doc_para_df = pd.concat([pd.read_json(AYMURAI_JSONS_PATH + fname) for fname in doc_para_files], ignore_index=True)

doc_df = doc_df.copy()
doc_para_df = doc_para_df.copy()
doc_df['id'] = doc_df['id'].astype(str)
doc_para_df['document_id'] = doc_para_df['document_id'].astype(str)

merged_df = pd.merge(
    doc_para_df,
    doc_df[['id', 'created_at', 'name']],
    left_on='document_id',
    right_on='id',
    how='left'
).rename(columns={'id_x': 'id'}).drop(columns=['id_y'])

df = pd.merge(
    para_df,
    merged_df,
    left_on='id',
    right_on='paragraph_id',
    how='left'
)

df = df[df['name'].fillna('').str.startswith('document') & (df['name']!= 'doc')]
df = df.rename(columns={'prediction':'NER_prediction'})
df.head(2)

## Load validation .docx

In [ ]:
docs2analize = ['2','3','4','6','7','8']
docs_file = [f for f in os.listdir(DOCS_PATH) if ('.docx' in f) and (len(set(docs2analize)&set(f))==1) ]

documents = {}
doc_paragraphs = {}
doc_start_end_chars = {}
joined_texts = {}
for d in docs_file:
    print(d)
    path = DOCS_PATH + d
    # Extract document
    document = extraction(path)
    # Construct paragraphs
    paragraphs = constract_paragraph(document)
    start_end_chars = take_start_end_paragraphs(paragraphs)
    documents[d] = document
    doc_paragraphs[d] = paragraphs
    joined_text = "\n".join(paragraphs)
    joined_texts[d] = joined_text
    doc_start_end_chars[d] = start_end_chars
    print(start_end_chars)
    print('\n')

In [ ]:
dfs = []
for doc, start_end_chars in doc_start_end_chars.items():
    df_chars = pd.DataFrame(start_end_chars)
    df_chars['doc'] = doc
    dfs.append(df_chars)
all_df_chars = pd.concat(dfs, ignore_index=True)

all_df_chars.head()

In [ ]:
main_df = df.merge(
    all_df_chars,
    left_on=['paragraph_id', 'name'],
    right_on=['paragraph_id', 'doc'],
    how='inner'
)
#main_df.to_csv('documents-02-08-sin05-conOpenAI.csv', index=False)
main_df.head()

In [ ]:
main_df[main_df['name']!=main_df['doc']]

### Save main_df (NER + validation) 

In [93]:
main_df.to_csv('df-NER-vals-02-08-sin05.csv', index=False)

## Load LLM prediction database .pkl

In [ ]:
if IS_LLM_FILE:
    try:
        with open(LLM_PREDICTION_FILE, "rb") as file:
            predictions = pickle.load(file)
    except Exception as e:
        print(f'There is no file named {LLM_PREDICTION_FILE}, please redefine LLM_PREDICTION_FILE variable')
else:
    print('There is no file of predictions')

In [ ]:
df_llm = main_df.copy()

#for d in set(df.name):
#d = 'document-04.docx'
#preds = predictions[d]
#df_d = df_openai[df_openai['name'] == d].reset_index(drop=True)
#df_d['prediction'] = df_d.apply(lambda row: get_paragraph_predictions(row, predictions), axis=1)

df_llm['prediction'] = df_llm.apply(lambda row: get_paragraph_predictions(row, predictions), axis=1)


In [ ]:
df_llm[df_llm['name']!=df_llm['doc']]

In [ ]:
df_llm.iloc[7].prediction[0]

{'label': 'DIRECCION',
 'text': 'Tacuarí 138, 7o Piso',
 'start_char': 18211,
 'end_char': 18231,
 'attrs': {},
 'alignment_status': <AlignmentStatus.MATCH_EXACT: 'match_exact'>}

### Save df_llm

In [ ]:
df_llm.to_json("documents-02-08-sin05-conLLM-check.json", orient="records", lines=True)
#df_loaded = pd.read_json("documents-02-08-sin05-conLLM-check.json", orient="records", lines=True)
df_llm.to_csv('documents-02-08-sin05-conLLM-check.csv', index=False)

In [101]:
df_loaded.iloc[7].prediction[0]

{'label': 'DIRECCION',
 'text': 'Tacuarí 138, 7o Piso',
 'start_char': 18211,
 'end_char': 18231,
 'attrs': {},
 'alignment_status': {'name': 'MATCH_EXACT', 'value': 'match_exact'}}

# Sanity check

In [ ]:
docs = list(set(df_llm.name))
docs.sort()

### Check number of paragraphs

In [ ]:
len_par_database = [{doc:len(doc_paragraphs[doc])} for doc in docs]
len_par_df = [{name: len(set(df_llm[df_llm['name']==name].paragraph_id))} for name in docs]
print(len_par_database)
print(len_par_df)


In [ ]:
print(len(set(doc_paragraphs['document-04.docx'])))
print(len(doc_paragraphs['document-04.docx']))

### Check all paragraphs in df

In [ ]:

for d in docs:
    pars = doc_paragraphs[d]
    df = df_llm[df_llm['name']==d]
    if are_all_par(pars,df):
        print(f'All paragraphs in {d} are in df_llm')

# Extract samples

In [109]:
examples = sample_cases(df_llm,'PER','prediction')
pprint(examples)

{
│   'TP': [
│   │   {
│   │   │   'paragraph_id': '4d0c831fc7c751729eb115c0a57be038',
│   │   │   'document': 'document-08.docx',
│   │   │   'parrafo': 'PINZON, HECTOR EZEQUIEL SOBRE 53 BIS - AGRAVANTES (CONDUCTAS DESCRIPTAS EN LOS ARTÍCULOS 51, 52 Y 53)',
│   │   │   'val': ['PINZON, HECTOR EZEQUIEL'],
│   │   │   'openai': ['PINZON, HECTOR EZEQUIEL'],
│   │   │   'ner': ['PINZON, HECTOR EZEQUIEL']
│   │   },
│   │   {
│   │   │   'paragraph_id': 'c06da9794c435ed5a662b30aeb77a762',
│   │   │   'document': 'document-07.docx',
│   │   │   'parrafo': 'I. HOMOLOGAR EL ACUERDO DE AVENIMIENTO formulado entre las partes y consecuentemente, CONDENAR a MARCELO GONZALEZ, DNI 23.875.846, en la presente causa No 31.972/2018, a la pena de TRES (3) AÑOS DE PRISIÓN EN SUSPENSO, por considerarlo autor penalmente responsable de los hechos que tuvieron lugar los días 7 de septiembre, 8 de septiembre, 19 de septiembre, 26 de septiembre, 28 de septiembre, entre el 5 y 6 de noviembre, 6 de noviembre, 9 de noviembre, 10 de noviembre, 13 de noviembre, 14 de noviembre, y entre el 15 y 19 de noviembre de 2018, constituyen los delitos de amenazas simples, amenazas agravadas por el uso de armas, amenazas coactivas, desobediencia e incendio con peligro para los bienes, los cuales concurren entre sí de forma real, los cuales tuvieron lugar en un CONTEXTO DE VIOLENCIA DE GÉNERO, debiendo cumplir durante el PLAZO DE TRES (3) AÑOS, con las siguientes REGLAS DE CONDUCTA consistentes en: 1) fijar domicilio en la calle Maza 1246, PB, dpto. "2", de esta Ciudad y comunicar cualquier modificación; 2) someterse al cuidado del Patronato de Liberados de la Ciudad, sito en la calle Coronel Díaz 2110, 5o Piso, de esta Ciudad, debiendo concurrir cada quince (15) días; 3) prohibición de contacto por cualquier medio con las tres víctimas, señoras NADIA QUIROGA, PAULA QUIROGA y BELEN QUIROGA, incluyendo las redes sociales; 4) prohibición de acercamiento a un radio menor de 500 mts. respecto de las nombradas y de sus respectivos domicilios, sitos en Olavarría 555 y de la peluquería sita en la calle Del Valle Iberlucea 500; 5) prohibición de referirse a NADIA QUIROGA, directa o indirectamente, sea a través de su nombre o de su imagen, a través de redes sociales y/o de cualquier otra plataforma digital, y obligación de dar de baja el contenido que él hubiera cargado en internet, sea en redes sociales o plataformas digitales, encomendado a la Fiscalía el control de esta regla; 6) realizar un tratamiento psicológico y/o psiquiátrico, previo dictamen por parte del Cuerpo Médico respecto de su necesidad y pertinencia; 7) realizar un taller vinculado con la prevención de la violencia de género "Con textos de violencia", dictado por Marisa Nazimof de la Subsecretaría de Derechos Humanos y Pluralismo Cultural de la Nación, en Av. Libertador 8151, debiendo acreditar su inscripción dentro del plazo de 10 días; con COSTAS (arts. 5, 26, 27 bis, 40, 41, 45 y 149 bis, primer párrafo, 149 bis, primer párrafo, segunda parte, 149 bis segundo párrafo, art. 239 y 186 inc. 1o CP, art. 55 CP; arts. 4, 5 inc. 1o y 2o y art. 6 de la Ley 26.485; y art. 1o de la Convención Interamericana para prevenir, sancionar y erradicar la violencia contra la mujer de "Belem do Pará"; y arts. 266 y 248 CPPCABA).',
│   │   │   'val': [
│   │   │   │   'MARCELO GONZALEZ',
│   │   │   │   'NADIA QUIROGA, PAULA QUIROGA',
│   │   │   │   'BELEN QUIROGA',
│   │   │   │   'NADIA QUIROGA',
│   │   │   │   'Marisa Nazimof'
│   │   │   ],
│   │   │   'openai': [
│   │   │   │   'MARCELO GONZALEZ',
│   │   │   │   'NADIA QUIROGA',
│   │   │   │   'PAULA QUIROGA',
│   │   │   │   'BELEN QUIROGA',
│   │   │   │   'NADIA QUIROGA',
│   │   │   │   'Marisa Nazimof'
│   │   │   ],
│   │   │   'ner': [
│   │   │   │   'MARCELO GONZALEZ',
│   │   │   │   'NADIA QUIROGA, PAULA QUIROGA',
│   │   │   │   'BELEN QUIROGA',
│   │   │   │   'NADIA QUIROGA',
│   │   │   │   'Marisa Nazimof'
│   │   │   ]
│   │   },
│   │   {
│   │   │

In [112]:
doc_paragraphs['document-04.docx']


['JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARIA N°19',
 'GOMEZ, ELVIS JUNIOR SOBRE 89 - LESIONES LEVES',
 'Número: IPP 8125/2020-0',
 'CUIJ: IPP J-01-00017381-5/2020-0',
 'Actuación Nro: 14544192/2020',
 'ACTA DE AUDIENCIA',
 'VIDEOCONFERENCIA',
 '"GOMEZ, ELVIS JUNIOR SOBRE 89 EN FUNCIÓN DEL 92, 149 BIS, 162 Y 239 DEL CÓDIGO PENAL"',
 'Causa N° 8125/2020',
 'Fecha: 4 de abril de 2020',
 'Horario de inicio: 15:00 horas',
 'Tipo de audiencia: audiencia de conocimiento personal (art. 266 CPPCABA)',
 'Juez: Pablo C. Casas -Juzgado Penal Contravencional y de Faltas Nro. 10-.',
 'Secretaria: Maria Agustina Iriarte López.',
 'PARTES PRESENTES',
 'Acusado: Elvis Junior GOMEZ, DNI n° 44.986.516.',
 'Defensa Oficial: Marina Recabarra, -Defensoría Oficial Nro. 20-.',
 'Fiscal: Adrián Dávila -Fiscalía Penal, Contravencional y de Faltas Nro. 36-.',
 'DESARROLLO',
 'Juez: Da inicio a la audiencia y explica que su objetivo es escuchar al acusado en virtud del acuerd